In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType
from itertools import chain
from pyspark.sql import Window

# Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.inventory_snapshot")

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

#Data Transform

##Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

##Standardize the date

In [0]:
(df.select(F.regexp_replace(F.regexp_replace(F.trim("snapshot_date"), "[0-9]", "d"), "[A-Za-z]", "a").alias("pattern"))
   .groupBy("pattern").count().orderBy(F.desc("count")).show(truncate=False))

In [0]:
df = df.withColumn(
    "snapshot_date",
    F.coalesce(
        F.try_to_date("snapshot_date", "yyyy-MM-dd"),
        F.try_to_date("snapshot_date", "yyyy-dd-MM")
    )
)

In [0]:
df.display()

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

##De-duplicate

In [0]:
dups = (df.groupBy(df.columns)
          .count()
          .filter("count > 1")
          .orderBy(F.desc("count")))

dups.show(1000, truncate=False)
print("duplicated groups:", dups.count())


w = Window.partitionBy(*df.columns).orderBy(F.lit(1))
df = df.withColumn("row_count", F.row_number().over(w))
df = df.filter("row_count = 1").drop("row_count") 

In [0]:
print("rows:", df.count())
print("columns:", len(df.columns))

#Write into silver table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.inventory_snapshot")
)

In [0]:
%sql
SELECT * FROM silver.inventory_snapshot

In [0]:
%sql
SELECT * FROM silver.inventory_snapshot
WHERE product_id = "P001"